# w13b — Model Selection in Python
**ENGR 1451/2451 · Exploratory Data Science · Fall 2026**

Use the full professor-evaluation data to compare **backward selection** rules. Then practice **one step of forward selection** with a small example. We do not make a training/test split or use prediction metrics here.

In [1]:
from pathlib import Path
import pandas as pd
import statsmodels.formula.api as smf

from pandas.api.types import is_numeric_dtype

## 1. Load the professor-evaluation data

Put your `evals.csv` file **next to this notebook** or inside a `data/` folder. The response is `score` (student evaluation, 1–5). Each row is a **course evaluation**, not a unique professor. The original study describes 463 courses taught by 94 professors, but this CSV does not contain a professor ID.

The lecture uses **nine predictor groups**: beauty, gender, age, outfit, class level, language, ethnicity, class size, and academic rank.

In [2]:
# Find the uploaded CSV in this folder or the data/ folder.
evals_file = Path('data/evals.csv')
evals = pd.read_csv(evals_file)
evals.head()

,score,rank,ethnicity,gender,language,age,cls_perc_eval,cls_did_eval,cls_students,cls_level,...,cls_credits,bty_f1lower,bty_f1upper,bty_f2upper,bty_m1lower,bty_m1upper,bty_m2upper,bty_avg,pic_outfit,pic_color
0,4.7,tenure track,minority,female,english,36,55.81395,24,43,upper,...,multi credit,5,7,6,2,4,6,5.0,not formal,color
1,4.1,tenure track,minority,female,english,36,68.80000,86,125,upper,...,multi credit,5,7,6,2,4,6,5.0,not formal,color
2,3.9,tenure track,minority,female,english,36,60.80000,76,125,upper,...,multi credit,5,7,6,2,4,6,5.0,not formal,color
3,4.8,tenure track,minority,female,english,36,62.60163,77,123,upper,...,multi credit,5,7,6,2,4,6,5.0,not formal,color
4,4.6,tenured,not minority,male,english,59,85.00000,17,20,upper,...,multi credit,4,4,2,2,3,3,3.0,not formal,color


In [3]:
exclude = {
    "score",           # Response variable
    "cls_perc_eval",
    "cls_did_eval",
    "cls_profs",
    "cls_credits",
    "pic_color",
}

# Exclude individual beauty ratings, but retain bty_avg
exclude.update(
    col for col in evals.columns
    if col.startswith("bty_") and col != "bty_avg"
)

# Automatically generate regression terms
terms = [
    col if is_numeric_dtype(evals[col]) else f"C({col})"
    for col in evals.columns
    if col not in exclude
]
terms

['C(rank)',
 'C(ethnicity)',
 'C(gender)',
 'C(language)',
 'age',
 'cls_students',
 'C(cls_level)',
 'bty_avg',
 'C(pic_outfit)']

## 2. Fit the full model

Use `smf.ols()` as in w13a. The three-level categorical variable `rank` requires **two indicator coefficients**, but it is still **one predictor group**. We will compare models using **adjusted $R^2$** (larger is better) and, later, a **$p$-value rule**.

The lecture's table quotes results from a historical analysis; use values computed from the **supplied CSV** for this exercise. They need not match every printed historical coefficient exactly.

In [11]:
def fit_terms(chosen):
    """Fit score on the specified predictor groups, using all rows."""
    formula = "score ~ " + (" + ".join(chosen) if chosen else "1")
    return smf.ols(formula, data=evals).fit()

full_model = fit_terms(terms)
print(f"Full model: adjusted R² = {full_model.rsquared_adj:.4f}; AIC = {full_model.aic:.1f}")
print(f"Predictor groups = {len(terms)}; slope coefficients = {int(full_model.df_model)}")
display(pd.DataFrame({"Coefficient": full_model.params,
                      "p-value": full_model.pvalues}).round(4))

Full model: adjusted R² = 0.0858; AIC = 719.3
Predictor groups = 9; slope coefficients = 10


,Coefficient,p-value
Intercept,4.4889,0.0000
C(rank)[T.tenure track],-0.1931,0.0228
C(rank)[T.tenured],-0.1571,0.0169
C(ethnicity)[T.not minority],0.0724,0.3426
C(gender)[T.male],0.2057,0.0001
C(language)[T.non-english],-0.2165,0.0594
C(cls_level)[T.upper],-0.0580,0.2945
C(pic_outfit)[T.not formal],-0.1497,0.0461
age,-0.0088,0.0070
cls_students,-0.0004,0.2952


## 3. Backward selection by adjusted $R^2$

**Start with the full model.** First try removing **each** predictor group, refitting the model every time. Keep a deletion only when it **increases adjusted $R^2$**. The factor `C(rank)` stays together as one group.

In [12]:
candidates = []
for dropped in terms:
    kept = [term for term in terms if term != dropped]
    candidates.append({
        "Remove": dropped,
        "Adjusted R²": fit_terms(kept).rsquared_adj,
    })

display(pd.DataFrame(candidates).sort_values("Adjusted R²", ascending=False).round(4))
print(f"Full-model adjusted R² = {full_model.rsquared_adj:.4f}")

,Remove,Adjusted R²
1,C(ethnicity),0.0860
5,cls_students,0.0856
6,C(cls_level),0.0856
3,C(language),0.0806
8,C(pic_outfit),0.0797
0,C(rank),0.0752
4,age,0.0730
7,bty_avg,0.0642
2,C(gender),0.0571


Full-model adjusted R² = 0.0858


### Repeat until no deletion improves the criterion

After **each** removal, repeat the comparison using the **newly refitted model**. Stepwise search is *greedy*: it does not examine every possible subset.

In [6]:
selected_adj = terms.copy()
current_adj = full_model.rsquared_adj
history_adj = [{"Step": 0, "Removed": "Start: full model", "Adjusted R²": current_adj}]

while selected_adj:
    choices = []
    for dropped in selected_adj:
        kept = [term for term in selected_adj if term != dropped]
        choices.append((fit_terms(kept).rsquared_adj, dropped, kept))
    best_adj, dropped, kept = max(choices, key=lambda row: row[0])
    if best_adj <= current_adj + 1e-10:
        break
    selected_adj = kept
    current_adj = best_adj
    history_adj.append({"Step": len(history_adj), "Removed": dropped,
                        "Adjusted R²": current_adj})

display(pd.DataFrame(history_adj).round(4))
print("Selected predictor groups:", selected_adj)

,Step,Removed,Adjusted R²
0,0,Start: full model,0.0858
1,1,C(ethnicity),0.0860
2,2,C(cls_level),0.0865
3,3,cls_students,0.0872


Selected predictor groups: ['C(rank)', 'C(gender)', 'C(language)', 'age', 'bty_avg', 'C(pic_outfit)']


## 4. Backward selection by $p$-values

The lecture's second rule starts with the full model. At $\alpha=0.05$:

1. Find the predictor **group** with the largest $p$-value **above 0.05**.
2. Remove that group, **refit**, and repeat.
3. Stop when no remaining group has a $p$-value above 0.05.

For an individual numerical predictor or a two-level categorical predictor, the comparison is equivalent to the ordinary slope test. For the three-level `rank` factor, use a **joint partial $F$-test** by comparing the current model with the model *without the entire term*: `full.compare_f_test(reduced)`. Do not pick just one rank-indicator $p$-value.

In [7]:
alpha = 0.05
selected_p = terms.copy()
history_p = [{"Step": 0, "Removed": "Start: full model", "p before removal": None}]

while selected_p:
    current_model = fit_terms(selected_p)
    group_pvalues = []
    for dropped in selected_p:
        kept = [term for term in selected_p if term != dropped]
        reduced_model = fit_terms(kept)
        f_stat, p_value, df_difference = current_model.compare_f_test(reduced_model)
        group_pvalues.append((p_value, dropped))
    largest_p, dropped = max(group_pvalues, key=lambda row: row[0])
    if largest_p <= alpha:
        break
    selected_p.remove(dropped)
    history_p.append({"Step": len(history_p), "Removed": dropped,
                      "p before removal": largest_p})

display(pd.DataFrame(history_p).round(4))
print("Selected predictor groups:", selected_p)

,Step,Removed,p before removal
0,0,Start: full model,NaN
1,1,C(ethnicity),0.3426
2,2,C(cls_level),0.3834
3,3,cls_students,0.4324
4,4,C(pic_outfit),0.0670
5,5,C(language),0.0563


Selected predictor groups: ['C(rank)', 'C(gender)', 'age', 'bty_avg']


### Compare the two selection rules

Different rules can keep different predictors. Here both rules begin with the **same full-data model**, and each decision is based on a **new fit**. Compare their adjusted $R^2$ and AIC; neither criterion makes an observational association causal.

For a concrete joint test, compare the final $p$-value-selected model with the same model **without `C(rank)`**.

In [8]:
p_model = fit_terms(selected_p)
no_rank = fit_terms([term for term in selected_p if term != "C(rank)"])
f_stat, p_rank, df_difference = p_model.compare_f_test(no_rank)
print(f"Joint test for rank (df difference = {int(df_difference)}): "
      f"F = {f_stat:.3f}; p = {p_rank:.4f}")

comparison = []
for name, chosen in [
    ("Full", terms),
    ("Backward adjusted R²", selected_adj),
    ("Backward p-value", selected_p),
]:
    fitted = fit_terms(chosen)
    comparison.append({"Model": name, "Groups": len(chosen),
                       "Adjusted R²": fitted.rsquared_adj, "AIC": fitted.aic})
display(pd.DataFrame(comparison).set_index("Model").round(4))

Joint test for rank (df difference = 2): F = 4.708; p = 0.0095


,Groups,Adjusted R²,AIC
Model,,,
Full,9,0.0858,719.2916
Backward adjusted R²,6,0.0872,715.6219
Backward p-value,4,0.0771,718.7415


## 5. Summary and Python function reference

- **Backward adjusted $R^2$:** Try deleting each predictor group; remove the one that increases adjusted $R^2$ most. Stop when no removal improves it.
- **Backward $p$-value rule:** Remove the least-significant predictor *group* when its joint $p$-value exceeds $0.05$; refit after every removal.
- **Forward adjusted $R^2$:** Start with only an intercept. Try adding each candidate group, then keep the one that increases adjusted $R^2$ most. Stop when no addition improves it.
- **Categorical variables:** `C(rank)` is treated as one predictor group even though it has multiple coefficients.

Useful Python: `smf.ols(formula, data=evals).fit()`, `.rsquared_adj`, `.aic`, `.compare_f_test(reduced_model)`.

Selection finds associations under a chosen criterion; it does **not** establish causation.

## 6. Forward selection: worked example

To see the idea without coding a full search algorithm, consider just **three candidates**: `bty_avg`, `age`, and `C(gender)`. Start with the **intercept-only** model, `score ~ 1`.

**Round 1:** Fit three one-predictor models. Which one has the largest adjusted $R^2$?

In [9]:
# Try adding each predictor to the intercept-only model.
mini_candidates = ["bty_avg", "age", "C(gender)"]
first_round = []

for term in mini_candidates:
    model = fit_terms([term])
    first_round.append({"Add": term, "Adjusted R²": model.rsquared_adj})

display(pd.DataFrame(first_round).sort_values("Adjusted R²", ascending=False).round(4))

,Add,Adjusted R²
0,bty_avg,0.0329
2,C(gender),0.0144
1,age,0.0093


**Result:** Add `bty_avg` first because its adjusted $R^2$ is highest. The model is now `score ~ bty_avg`.

**Round 2:** Keep `bty_avg` in the model and consider adding **one** of the two remaining terms. That is your in-class exercise.

## 7. Your turn

1. Compare adding `age` versus adding `C(gender)` to a model that already contains `bty_avg`.
2. **Choose one:** Which predictor would you add next, and what is the new adjusted $R^2$?
3. **Discuss:** Does choosing a predictor mean it *causes* higher student evaluation scores? Explain in one sentence.

You do **not** need to program the whole forward-selection algorithm.

In [10]:

# Write your responses to questions 2–3 in a Markdown cell.

Add age: adjusted R² = 0.0336
Add C(gender): adjusted R² = 0.0550
